In [7]:
import pandas as pd

# 파일의 정확한 경로와 확장자(.xlsx)를 지정합니다.
file_path = r'F:\Download\Rawdata.xlsx'

try:
    # 엑셀 파일 로딩 (openpyxl 엔진 사용)
    df = pd.read_excel(file_path, engine='openpyxl')
    print("✅ 엑셀 파일을 성공적으로 불러왔습니다.\n")
    
    # 상위 5개 행 출력
    print(df.head())
    
except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{file_path}' 경로를 확인해주세요.")
except ImportError:
    print("❌ 엑셀 파일을 읽기 위한 'openpyxl' 라이브러리가 설치되어 있지 않습니다.")
    print("💡 터미널(또는 명령 프롬프트)에서 'pip install openpyxl'을 입력하여 설치해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

✅ 엑셀 파일을 성공적으로 불러왔습니다.

  patient_id 사고 및 질환 항목  연령대  source_count Gender
0    P000001  악성신생물 (암)  10대             1      M
1    P000002  악성신생물 (암)  10대             1      M
2    P000003  악성신생물 (암)  10대             1      M
3    P000004  악성신생물 (암)  10대             1      M
4    P000005  악성신생물 (암)  10대             1      M


In [8]:
import pandas as pd
import numpy as np

# 1. 5대 분류 매핑 함수 정의
def classify_risk(item):
    """
    '사고 및 질환 항목'을 5대 핵심 담보 카테고리로 분류하는 함수
    """
    # 결측치(NaN) 방어 코드
    if pd.isna(item) or not isinstance(item, str):
        return '기타/미분류'

    # [Category 1] 면책 항목 (보험 보장 제외 대상)
    exemptions = [
        '고의적 자해(자살)', 
        '정신활성 물질 사용에 의한 정신 및 행동 장애'
    ]
    
    # [Category 2] 일반 상해 항목 (추락 제외)
    injuries = [
        '운수 사고 (교통사고)', 
        '불의의 물에 빠짐 (익사)', 
        '유독성 물질에 의한 불의의 중독 및 노출', 
        '가해 (타살)', 
        '연기, 불 및 불꽃에 노출'
    ]
    
    # --- 조건부 매핑 로직 ---
    if item in exemptions:
        return '면책'
    elif item == '추락 (미끄러짐)':
        return '3대 비급여'
    elif item in injuries:
        return '상해통원'
    else:
        return '질병통원'

# ==========================================
# 2. 데이터 불러오기 및 분류 적용
# ==========================================

# 원본 파일 경로 및 저장할 새 파일 경로 설정
input_file_path = r'F:\Download\Rawdata.xlsx'
output_file_path = r'F:\Download\Rawdata_Classified.xlsx'

try:
    print("⏳ 데이터를 불러오는 중입니다...")
    # 엑셀 파일 로딩
    df = pd.read_excel(input_file_path, engine='openpyxl')
    
    print("⚙️ 새로운 분류 컬럼을 생성하는 중입니다...")
    # '사고 및 질환 항목' 컬럼에 classify_risk 함수를 적용하여 '담보_분류_결과'라는 새 컬럼 생성
    df['담보_분류_결과'] = df['사고 및 질환 항목'].apply(classify_risk)
    
    print("💾 분류된 데이터를 새로운 엑셀 파일로 저장하는 중입니다...")
    # 결과를 새로운 엑셀 파일로 저장 (원본 파일 보호)
    df.to_excel(output_file_path, index=False, engine='openpyxl')
    
    # 최종 결과 요약 출력
    print("\n✅ 작업이 완료되었습니다!")
    print(f"📁 저장된 파일 위치: {output_file_path}")
    print("\n📊 [새로 생성된 담보 분류 분포]")
    print(df['담보_분류_결과'].value_counts())
    
    # 상위 5개 행 미리보기 (해당 컬럼들만)
    print("\n🔍 [데이터 미리보기]")
    print(df[['사고 및 질환 항목', '담보_분류_결과']].head())

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{input_file_path}' 경로를 확인해주세요.")
except KeyError:
    print("❌ 엑셀 파일 내에 '사고 및 질환 항목'이라는 이름의 컬럼이 없습니다. 컬럼명을 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오는 중입니다...
⚙️ 새로운 분류 컬럼을 생성하는 중입니다...
💾 분류된 데이터를 새로운 엑셀 파일로 저장하는 중입니다...

✅ 작업이 완료되었습니다!
📁 저장된 파일 위치: F:\Download\Rawdata_Classified.xlsx

📊 [새로 생성된 담보 분류 분포]
담보_분류_결과
질병통원      56636
면책        12575
상해통원       2173
3대 비급여     1027
Name: count, dtype: int64

🔍 [데이터 미리보기]
  사고 및 질환 항목 담보_분류_결과
0  악성신생물 (암)     질병통원
1  악성신생물 (암)     질병통원
2  악성신생물 (암)     질병통원
3  악성신생물 (암)     질병통원
4  악성신생물 (암)     질병통원


In [10]:
import pandas as pd
import numpy as np

# 파일 경로 설정
input_file_path = r'F:\Download\Rawdata_Classified.xlsx'
output_file_path = r'F:\Download\Step3_Risk_Score_Table_by_Age.xlsx'

try:
    print("⏳ 데이터를 불러오고 있습니다...")
    df = pd.read_excel(input_file_path, engine='openpyxl')
    
    # 1. 연령대별 담보 분류 교차표(Cross Table) 생성
    print("⚙️ 연령대별 질병 발생 빈도 및 확률을 계산 중입니다...")
    risk_crosstab = pd.crosstab(df['연령대'], df['담보_분류_결과'])
    
    # 2. 연령대별 전체 사고 건수 계산
    risk_crosstab['총_사고건수'] = risk_crosstab.sum(axis=1)
    
    # '질병통원' 컬럼이 존재하는지 확인 후 로직 실행
    if '질병통원' in risk_crosstab.columns:
        # 3. 질병 발생 확률 (연령대별 질병건수 / 해당 연령대 총 사고건수)
        risk_crosstab['질병_발생확률'] = risk_crosstab['질병통원'] / risk_crosstab['총_사고건수']
        
        # 4. 가중치 정규화 (Min-Max Scaling 변형: 가장 확률이 높은 연령대를 1.0으로 설정)
        max_disease_rate = risk_crosstab['질병_발생확률'].max()
        risk_crosstab['Disease_Risk_Score'] = (risk_crosstab['질병_발생확률'] / max_disease_rate).round(4)
        
        # (보너스) 상해 가중치 스코어도 동일한 방식으로 계산하여 모델의 완성도를 높임
        if '상해통원' in risk_crosstab.columns:
            risk_crosstab['상해_발생확률'] = risk_crosstab['상해통원'] / risk_crosstab['총_사고건수']
            max_injury_rate = risk_crosstab['상해_발생확률'].max()
            risk_crosstab['Injury_Risk_Score'] = (risk_crosstab['상해_발생확률'] / max_injury_rate).round(4)
            
            # 최종 표 정리를 위한 컬럼 선택
            final_table = risk_crosstab[['총_사고건수', '질병통원', '질병_발생확률', 'Disease_Risk_Score', 
                                         '상해통원', '상해_발생확률', 'Injury_Risk_Score']]
        else:
            final_table = risk_crosstab[['총_사고건수', '질병통원', '질병_발생확률', 'Disease_Risk_Score']]
            
        # 질병 위험도가 높은 순으로 정렬
        final_table = final_table.sort_values(by='Disease_Risk_Score', ascending=False)
        
        # 5. 결과 출력 및 엑셀 저장
        print("\n✅ [연령대별 질병 가중치 스코어 표 (Disease Risk Score)]")
        print(final_table)
        
        final_table.to_excel(output_file_path, engine='openpyxl')
        print(f"\n💾 스코어 테이블이 성공적으로 저장되었습니다: {output_file_path}")
        
    else:
        print("❌ 분류된 데이터에 '질병통원' 항목이 존재하지 않습니다.")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{input_file_path}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오고 있습니다...
⚙️ 연령대별 질병 발생 빈도 및 확률을 계산 중입니다...

✅ [연령대별 질병 가중치 스코어 표 (Disease Risk Score)]
담보_분류_결과  총_사고건수   질병통원   질병_발생확률  Disease_Risk_Score  상해통원   상해_발생확률  \
연령대                                                                     
60-69세     36593  32834  0.897275              1.0000   803  0.021944   
50-59세     20157  15861  0.786873              0.8770   484  0.024012   
40-49세      8651   5327  0.615767              0.6863   289  0.033407   
10대         1249    713  0.570857              0.6362   153  0.122498   
30-39세      3700   1445  0.390541              0.4353   225  0.060811   
20-29세      2061    456  0.221252              0.2466   219  0.106259   

담보_분류_결과  Injury_Risk_Score  
연령대                          
60-69세               0.1791  
50-59세               0.1960  
40-49세               0.2727  
10대                  1.0000  
30-39세               0.4964  
20-29세               0.8674  

💾 스코어 테이블이 성공적으로 저장되었습니다: F:\Download\Step3_Risk_Score_Table_by_Age.xlsx


In [11]:
import pandas as pd

# 1. 파일 경로 설정
input_file_path = r'F:\Download\Rawdata_Classified.xlsx'
output_file_path = r'F:\Download\Step3_Coverage_Risk_Score_Table.xlsx'

try:
    print("⏳ 데이터를 불러오는 중입니다...")
    df = pd.read_excel(input_file_path, engine='openpyxl')
    
    # '담보_분류_결과' 컬럼 존재 여부 확인
    if '담보_분류_결과' not in df.columns:
        raise ValueError("'담보_분류_결과' 컬럼이 존재하지 않습니다. 먼저 분류 코드를 실행했는지 확인해주세요.")
    
    print("⚙️ 담보별 빈도 및 가중치 스코어를 계산하는 중입니다...")
    
    # 2. 담보 카테고리별 발생 빈도(건수) 계산
    coverage_counts = df['담보_분류_결과'].value_counts().reset_index()
    coverage_counts.columns = ['담보_카테고리', '발생_건수']
    
    # 전체 사고 건수 (면책, 기타 포함 전체)
    total_cases = coverage_counts['발생_건수'].sum()
    
    # 3. 담보별 발생 확률(비중) 계산
    coverage_counts['발생_비중(%)'] = (coverage_counts['발생_건수'] / total_cases * 100).round(2)
    
    # 4. 가중치 정규화 (가장 빈도가 높은 카테고리를 1.0으로 설정)
    # 참고: '면책'이나 '기타/미분류'는 보험 모델링의 기준점이 되기 부적절할 수 있으나, 
    # 보통 질병통원이나 상해통원이 1위이므로 그대로 max()를 적용합니다.
    max_count = coverage_counts['발생_건수'].max()
    coverage_counts['Coverage_Risk_Score'] = (coverage_counts['발생_건수'] / max_count).round(4)
    
    # 5. 결과 정렬 및 출력
    # 스코어가 높은 순(발생 빈도가 높은 순)으로 정렬
    final_table = coverage_counts.sort_values(by='Coverage_Risk_Score', ascending=False)
    
    print("\n✅ [주요 담보별 위험 가중치 스코어 산출 결과]")
    print("-" * 60)
    print(final_table.to_string(index=False))
    print("-" * 60)
    
    # 6. 결과를 새로운 엑셀 파일로 저장
    final_table.to_excel(output_file_path, index=False, engine='openpyxl')
    print(f"\n💾 담보별 스코어 테이블이 성공적으로 저장되었습니다:\n   👉 {output_file_path}")

except FileNotFoundError:
    print(f"❌ 파일을 찾을 수 없습니다. '{input_file_path}' 경로를 확인해주세요.")
except Exception as e:
    print(f"❌ 오류가 발생했습니다: {e}")

⏳ 데이터를 불러오는 중입니다...
⚙️ 담보별 빈도 및 가중치 스코어를 계산하는 중입니다...

✅ [주요 담보별 위험 가중치 스코어 산출 결과]
------------------------------------------------------------
담보_카테고리  발생_건수  발생_비중(%)  Coverage_Risk_Score
   질병통원  56636     78.21               1.0000
     면책  12575     17.37               0.2220
   상해통원   2173      3.00               0.0384
 3대 비급여   1027      1.42               0.0181
------------------------------------------------------------

💾 담보별 스코어 테이블이 성공적으로 저장되었습니다:
   👉 F:\Download\Step3_Coverage_Risk_Score_Table.xlsx
